In [ ]:
import pandas as pd
import polars as pl
import sqlite3
import os
import gc
from datetime import datetime

# --- CONFIGURATION ---
SELECTED_COMMUNITY = 'Stack Overflow'  # Options: 'Stack Overflow', 'Mathematics', 'Ask Ubuntu'

# Paths
SOURCE_DB_PATH = {
    'Stack Overflow': r'E:\Stack Overflow.db',
    'Mathematics': r'E:\Mathematics.db',
}.get(SELECTED_COMMUNITY)

# Path to the User Tracking DB created in the previous step
TRACKING_DB_PATH = f"E:\\{SELECTED_COMMUNITY}_User_Tracking.db" 

# Output Filenames (REMOVED 'New User')
OUTPUT_FILES = {
    'Super': f"{SELECTED_COMMUNITY}_Daily_Super.csv",
    'Regular': f"{SELECTED_COMMUNITY}_Daily_Regular.csv",
    'Casual': f"{SELECTED_COMMUNITY}_Daily_Casual.csv"
}

# --- ROBUST TEXT METRICS ---
def calculate_text_metrics(df):
    # 1. Fill Nulls
    df = df.with_columns(pl.col("Body").fill_null("").alias("Body"))

    # 2. Text Length
    df = df.with_columns(
        pl.col("Body").str.len_chars().cast(pl.Float64).alias("Text_Length")
    )

    # 3. Code Length
    pattern = r"(?s)<code>.*?</code>|<pre>.*?</pre>|\$\$.*?\$\$|\$.*?\$"
    df = df.with_columns(
        pl.col("Body").str.replace_all(pattern, "").str.len_chars().cast(pl.Float64).alias("Non_Code_Length")
    )
    df = df.with_columns(
        (pl.col("Text_Length") - pl.col("Non_Code_Length")).fill_nan(0.0).alias("Code_Length")
    )

    # 4. Readability
    clean_body = pl.col("Body").str.replace_all(r"<[^>]+>", " ")
    df = df.with_columns([
        clean_body.str.count_matches(r"[.!?]").cast(pl.Float64).alias("n_sentences"),
        clean_body.str.count_matches(r"\s+").cast(pl.Float64).alias("n_words"),
        clean_body.str.count_matches(r"(?i)[aeiouy]").cast(pl.Float64).alias("n_vowels")
    ])
    df = df.with_columns(
        (pl.col("n_words") / (pl.col("n_sentences") + 1.0)).alias("avg_sent_len"),
        (pl.col("n_vowels") / (pl.col("n_words") + 1.0)).alias("avg_word_len")
    )
    df = df.with_columns(
        (206.835 - (1.015 * pl.col("avg_sent_len")) - (84.6 * pl.col("avg_word_len")))
        .fill_nan(0.0).alias("Readability_Score")
    )

    cols_to_keep = [
        "CreationDate", "PostTypeId", "OwnerUserId", "Score", 
        "ViewCount", "AnswerCount", "CommentCount", 
        "AcceptedAnswerId", "ClosedDate", 
        "Text_Length", "Code_Length", "Readability_Score"
    ]
    
    return df.select([c for c in cols_to_keep if c in df.columns])

# --- HELPER: LOAD USER TRACKING DATA ---
def load_user_tracking_map():
    """
    Loads the User Tracking DB into a Polars LazyFrame or DataFrame 
    to allow joining on (Day, UserId).
    """
    if not os.path.exists(TRACKING_DB_PATH):
        print(f"CRITICAL ERROR: Tracking DB not found at {TRACKING_DB_PATH}")
        return None

    print("Loading User Tracking Database into memory...")
    conn = sqlite3.connect(TRACKING_DB_PATH)
    
    # We only need Day, UserId, and User_Classification
    query = "SELECT Day, UserId, User_Classification FROM User_Daily_Status"
    
    tracking_df = pl.read_database(query, conn).with_columns([
        pl.col("Day").str.to_datetime("%Y-%m-%d"), 
        pl.col("UserId").cast(pl.Int64)
    ])
    conn.close()
    
    # Rename for easier join
    tracking_df = tracking_df.rename({"UserId": "OwnerUserId", "User_Classification": "User_Class"})
    
    print(f"Tracking Data Loaded: {tracking_df.height} rows.")
    return tracking_df

# --- MAIN SCRIPT ---
def create_split_datasets():
    if not os.path.exists(SOURCE_DB_PATH): 
        print(f"Error: Database not found at {SOURCE_DB_PATH}")
        return

    # 1. Load User Classifications
    tracking_df = load_user_tracking_map()
    if tracking_df is None: return

    conn = sqlite3.connect(SOURCE_DB_PATH)
    
    # 2. Query
    query = """
    SELECT 
        CreationDate, PostTypeId, OwnerUserId, 
        Score, ViewCount, AnswerCount, CommentCount, 
        AcceptedAnswerId, ClosedDate, Body
    FROM Posts
    WHERE CreationDate >= '2018-01-01'
    """
    
    chunk_size = 50000 
    aggregates = [] 
    
    print(f"Streaming data from {SELECTED_COMMUNITY}...")
    reader = pd.read_sql_query(query, conn, chunksize=chunk_size)
    
    for i, pdf in enumerate(reader):
        print(f"Processing chunk {i+1}...", end='\r')
        
        # A. Convert to Polars & Cast
        df = pl.from_pandas(pdf).with_columns([
            pl.col("PostTypeId").cast(pl.Int64),
            pl.col("OwnerUserId").cast(pl.Int64), # Vital for Join
            pl.col("Score").cast(pl.Float64).fill_null(0.0),
            pl.col("ViewCount").cast(pl.Float64).fill_null(0.0),
            pl.col("AnswerCount").cast(pl.Float64).fill_null(0.0),
            pl.col("CommentCount").cast(pl.Float64).fill_null(0.0),
            pl.col("CreationDate").str.to_datetime(),
            pl.col("ClosedDate").str.to_datetime()
        ])
        
        # B. Text Metrics
        df = calculate_text_metrics(df)
        
        # C. Create Join Key 'Day'
        df = df.with_columns(
            pl.col("CreationDate").dt.truncate("1d").alias("Day")
        )

        # --- D. JOIN WITH USER TRACKING ---
        df = df.join(tracking_df, on=["Day", "OwnerUserId"], how="left")
        
        # --- E. STRICT FILTER: REMOVE NEW USERS / UNKNOWN ---
        # We only keep rows where the user class is exactly one of the 3 target types.
        # This removes any nulls (failed joins) or unexpected labels.
        valid_classes = ["Super", "Regular", "Casual"]
        df = df.filter(pl.col("User_Class").is_in(valid_classes))
        
        # F. Helper Columns
        df = df.with_columns([
            (pl.col("PostTypeId") == 1).alias("Is_Question"),
            (pl.col("PostTypeId") == 2).alias("Is_Answer"),
            
            # Metric splits (Q vs A)
            (pl.when(pl.col("PostTypeId") == 1).then(pl.col("Text_Length")).otherwise(0.0)).alias("Len_Q"),
            (pl.when(pl.col("PostTypeId") == 1).then(pl.col("Code_Length")).otherwise(0.0)).alias("CodeLen_Q"),
            (pl.when(pl.col("PostTypeId") == 2).then(pl.col("Text_Length")).otherwise(0.0)).alias("Len_A"),
            (pl.when(pl.col("PostTypeId") == 2).then(pl.col("Code_Length")).otherwise(0.0)).alias("CodeLen_A"),

            # Unanswered
            (pl.when((pl.col("PostTypeId") == 1) & (pl.col("AnswerCount") == 0))
             .then(pl.col("Text_Length")).otherwise(0.0)).alias("Len_Unanswered_Q"),
            (pl.when((pl.col("PostTypeId") == 1) & (pl.col("AnswerCount") == 0))
             .then(pl.col("Code_Length")).otherwise(0.0)).alias("CodeLen_Unanswered_Q"),
            (pl.when((pl.col("PostTypeId") == 1) & (pl.col("AnswerCount") == 0))
             .then(1.0).otherwise(0.0)).alias("Count_Unanswered_Q"),

            # Accepted
            (pl.when((pl.col("PostTypeId") == 1) & (pl.col("AcceptedAnswerId").is_not_null()))
             .then(pl.col("AnswerCount")).otherwise(0.0)).alias("Answers_on_Accepted_Q"),
            (pl.when((pl.col("PostTypeId") == 1) & (pl.col("AcceptedAnswerId").is_not_null()))
             .then(1.0).otherwise(0.0)).alias("Count_Accepted_Q"),
            (pl.when((pl.col("PostTypeId") == 1) & (pl.col("AcceptedAnswerId").is_not_null()))
             .then(pl.col("Text_Length")).otherwise(0.0)).alias("Len_Q_with_Accepted"),
            (pl.when((pl.col("PostTypeId") == 1) & (pl.col("AcceptedAnswerId").is_not_null()))
             .then(pl.col("Code_Length")).otherwise(0.0)).alias("CodeLen_Q_with_Accepted"),
        ])
        
        # G. Aggregate by Day AND User_Class
        agg = df.group_by(["Day", "User_Class"]).agg([
            pl.len().cast(pl.Float64).alias("Total_Posts"),
            pl.col("Is_Question").sum().alias("Total_Questions"),
            pl.col("Is_Answer").sum().alias("Total_Answers"),
            
            pl.col("Len_Q").sum().alias("Sum_Len_Q"),
            pl.col("CodeLen_Q").sum().alias("Sum_CodeLen_Q"),
            pl.col("Len_A").sum().alias("Sum_Len_A"),
            pl.col("CodeLen_A").sum().alias("Sum_CodeLen_A"),
            
            pl.col("Count_Unanswered_Q").sum().alias("Total_Unanswered_Q"),
            pl.col("Len_Unanswered_Q").sum().alias("Sum_Len_Unanswered_Q"),
            pl.col("CodeLen_Unanswered_Q").sum().alias("Sum_CodeLen_Unanswered_Q"),
            
            pl.col("Count_Accepted_Q").sum().alias("Total_Accepted_Q"),
            pl.col("Answers_on_Accepted_Q").sum().alias("Sum_Answers_on_Accepted_Q"),
            pl.col("Len_Q_with_Accepted").sum().alias("Sum_Len_Q_with_Accepted"),
            pl.col("CodeLen_Q_with_Accepted").sum().alias("Sum_CodeLen_Q_with_Accepted"),
            
            pl.col("Score").sum().alias("Sum_Score"),
            pl.col("ViewCount").sum().alias("Sum_Views"),
            pl.col("CommentCount").sum().alias("Sum_Comments"), 
            pl.col("Readability_Score").sum().alias("Sum_Readability"),
            pl.col("OwnerUserId").n_unique().cast(pl.Float64).alias("Active_Users")
        ])
        
        aggregates.append(agg)
        
        del pdf, df
        gc.collect()

    conn.close()
    
    # 3. Final Merge
    if not aggregates:
        print("No data found.")
        return

    print("\nMerging chunks and saving files...")
    full_df = pl.concat(aggregates)
    
    # Consolidate
    final = full_df.group_by(["Day", "User_Class"]).agg(pl.all().sum()).sort("Day")
    
    # 4. Split and Save (Iterate only through Valid Classes)
    output_cols = [
        "Day", 
        "Total_Posts", "Total_Questions", "Total_Answers", "Active_Users",
        "Total_Unanswered_Q", "Total_Accepted_Q",
        "Sum_Len_Q", "Sum_CodeLen_Q",
        "Sum_Len_A", "Sum_CodeLen_A",
        "Sum_Len_Unanswered_Q", "Sum_CodeLen_Unanswered_Q",
        "Sum_Len_Q_with_Accepted", "Sum_CodeLen_Q_with_Accepted",
        "Sum_Answers_on_Accepted_Q",
        "Sum_Score", "Sum_Views", "Sum_Comments", "Sum_Readability"
    ]
    
    classes = ["Super", "Regular", "Casual"]
    
    for cls in classes:
        print(f"Saving {cls} data...")
        class_df = final.filter(pl.col("User_Class") == cls)
        
        if class_df.height > 0:
            target_file = OUTPUT_FILES.get(cls, f"{SELECTED_COMMUNITY}_Daily_{cls}.csv")
            class_df.select(output_cols).write_csv(target_file)
        else:
            print(f"Warning: No data found for class {cls}")

    print("Done! 3 Files Created (Super, Regular, Casual).")

if __name__ == "__main__":
    create_split_datasets()

Loading User Tracking Database into memory...
Tracking Data Loaded: 621005 rows.
Streaming data from Mathematics...
Processing chunk 37...
Merging chunks and saving files...
Saving Super data...
Saving Regular data...
Saving Casual data...
Done! 3 Files Created (Super, Regular, Casual).
